<!-- <style> * { font-size: 100%; } </style> -->
<small> 

# rsyncd
- https://linux.die.net/man/5/rsyncd.conf
- https://superuser.com/questions/243656/how-to-configure-and-use-rsyncd


</small>

In [ ]:
## Setup Working Directory & Files

# Working Dir
WORKING_DIRECTORY="/mnt/user/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# Make Folders & Files
# mkdir -p ./{config,repositories}
mkdir -p ./{config,secrets}
ls -alt ${WORKING_DIRECTORY}

# Create Needed Files
touch ${WORKING_DIRECTORY}/secrets/rsyncd.secrets

# Docker

## Docker Context

In [ ]:
# Setup Docker on Remote Host via Context
# docker context create lxc-docker-01 --docker "host=ssh://lxc-docker-01"

set -euo pipefail

# SSH Alias Name (/config/.ssh/config)
CONTEXTS=(
  "nas-00"
  "nas-01"
  "nas-03"
  "lxc-docker-01"
  "lxc-docker-02"
  "lxc-docker-03"
  "lxc-docker-04"
  "lxc-docker-05"
  "lxc-docker-06"
  "lxc-docker-07"
  "lxc-docker-08"
  "lxc-docker-09"
)

for CONTEXT in "${CONTEXTS[@]}"; do
  if docker context inspect "${CONTEXT}" >/dev/null 2>&1; then
    printf 'Context %s : already exists\n' "${CONTEXT}"
  else
    docker context create "${CONTEXT}" \
      --docker "host=ssh://${CONTEXT}" \
      >/dev/null 2>&1
    printf 'Context %s : created successfully\n' "${CONTEXT}"
  fi
done

In [ ]:
# Remove Context
# docker context remove lxc-docker-06
# docker context remove lxc-docker-07

In [ ]:
# Use Context
# docker context use default
# docker context use lxc-docker-01
docker context use nas-03

## Build and Deploy Container

In [ ]:
# Build
docker compose --file docker-compose.yaml --env-file docker-compose.example.env build

In [ ]:
# Setup
WORKING_DIRECTORY="/mnt/user/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# mkdir -p ./{secrets}
mkdir -p ./secrets
ls -alt ${WORKING_DIRECTORY}

cat > ${WORKING_DIRECTORY}/secrets/rsyncd.secrets <<EOF
# Username must match RSYNCD_AUTH_USER in .env
synobackup:replace-with-a-long-random-password
EOF

In [ ]:
# Run
docker compose --file docker-compose.yaml --env-file docker-compose.example.env up --detach --build

In [ ]:
# Create the Minecraft Bedrock Server & Backup Service
CONTEXT=nas-03
# docker --context lxc-docker-01 compose --profile "*" --env-file .env.lxc-docker  up --build --detach --remove-orphans

# Build the Base Image for Minecraft Bedrock Server Backup (Bedrockifier)
docker --context "${CONTEXT}" build \
  --progress plain \
  --tag local/bedrockifier-base:latest \
  --file ../repositories/minecraft-bedrock-backup/Docker/Dockerfile \
  ../repositories/minecraft-bedrock-backup

# Build + Run the Minecraft Bedrock Server & Backup Service
docker --context "${CONTEXT}" compose \
  --ansi never \
  --env-file ../.env.lxc-docker \
  up --build --detach --remove-orphans